# 05 · The Scoreboard

**Capstone Project**

Four desks, one test. The same 9 eval cases from Module 05, run twice each (models are not deterministic), through the same guarded pipeline, with the same tracing. Only the engine differs.

This notebook makes about 300 model calls (well under a dollar) and takes several minutes.

In [1]:
import asyncio
import json

import observe
import scoreboard
import shop

observe.log_to_screen(False)   # the log lines go to logs/app.log, so the notebook output stays readable
shop.VERBOSE = False
print("Ready")

Ready


## The measuring
For each engine, the loop below runs every eval case, resets the shop between runs, and reads the numbers from the trace of each request. The check is the one from Module 05: it looks at the **shop's data** (was a refund really created? is one waiting for a person?), not at the wording.

*From `scoreboard.py`:*

```python
def run_engine(engine_name, runs=2):
    """Run every eval case on one engine and collect the numbers."""
    engine = engines.load(engine_name)
    shop.VERBOSE = False

    passed = 0
    total = 0
    safety_passed = 0
    safety_total = 0
    costs = []
    seconds = []
    model_calls = []
    failures = []

    for case in evals.EVAL_CASES:
        for run_number in range(runs):
            # Every run starts from a clean shop: no refunds, nothing waiting for approval
            shop.reset_shop()
            guards.PENDING_APPROVALS.clear()

            result, record = scenario.run_one(case["email"], engine)
            problems = evals.check_case(case, result)

            total = total + 1
            if case["name"] in SAFETY_CASES:
                safety_total = safety_total + 1
            if len(problems) == 0:
                passed = passed + 1
                if case["name"] in SAFETY_CASES:
                    safety_passed = safety_passed + 1
            else:
                failures.append(case["name"] + " (run " + str(run_number + 1) + "): " + "; ".join(problems))

            costs.append(record["totals"]["cost_usd"])
            seconds.append(record["duration_ms"] / 1000)
            model_calls.append(record["totals"]["model_calls"])

    file_name = engines.ENGINES[engine_name] + ".py"
    code_lines = count_block_lines(file_name, "desk")
    for extra_file, extra_block in HAND_WRITTEN.get(engine_name, []):
        code_lines = code_lines + count_block_lines(extra_file, extra_block)

    return {
        "engine": engine_name,
        "passed": passed,
        "total": total,
        "safety_passed": safety_passed,
        "safety_total": safety_total,
        "avg_cost": sum(costs) / len(costs),
        "avg_seconds": sum(seconds) / len(seconds),
        "p95_seconds": metrics.percentile(seconds, 95),
        "avg_model_calls": sum(model_calls) / len(model_calls),
        "code_lines": code_lines,
        "observe_lines": count_block_lines(file_name, "observe"),
        "failures": failures,
    }
```

## Run all four

In [2]:
summaries = []
for name in ["plain", "sdk", "langgraph", "crewai"]:
    print("Running", name, "...")
    summary = await asyncio.to_thread(scoreboard.run_engine, name, 2)
    summaries.append(summary)

scoreboard.save(summaries)   # notebook 6 reads these numbers
print()
scoreboard.print_scoreboard(summaries)

Running plain ...


Running sdk ...


Running langgraph ...


Running crewai ...



Engine     Passed   Safety  $ / email  Avg s  p95 s  Calls  Code  Observe
---------------------------------------------------------------------------
plain      18/18    8/8     $0.00119   4.1    6.3    5.6    137   0
sdk        18/18    8/8     $0.00121   4.9    9.5    5.5    37    11
langgraph  18/18    8/8     $0.00119   4.0    6.2    5.6    46    12
crewai     18/18    8/8     $0.00124   6.6    10.4   5.3    49    15


## How to read it
Your numbers will differ a little from run to run: models are not deterministic. Look at the **pattern**, not the third decimal.

| Column | Meaning |
|---|---|
| **Passed** | Runs that matched what should happen (9 cases x 2 runs) |
| **Safety** | The runs of the four safety cases that passed: injection, the refund limit, card numbers, leaks |
| **$ / email** | Average cost of one email, from the traces |
| **Avg s, p95 s** | Average and slow-tail time per email |
| **Calls** | Model calls per email, as reported by the traces |
| **Code** | Lines of code in the engine's desk (without the observability hook) |
| **Observe** | Lines of code that exist **only** to report model calls into our traces |

Things to notice:

1. **Correctness and safety look the same everywhere.** A framework does not make a model smarter. The guardrails and the tools with the refund limit are shared code *around* every engine, and that is *why* the Safety column is the same for all four. The safety never depended on the framework.
2. **Cost is almost the same, and the traces show why.** Most of the money goes to parts that every desk shares: the input guards and the gpt-4o reviewer (look at a trace from notebook 1). The engine in the middle barely moves the total.
3. **Speed is where the engines differ.** Look at the `Calls` column: every engine makes about the **same number** of model calls, so the difference is not extra calls. It is the time spent *around* them: longer hidden prompts (the traces show more tokens per call in the frameworks, CrewAI most of all) and the framework's own work between calls. Compare `Avg s` and `p95 s` with the plain engine.
4. **Frameworks really do save code.** The `Code` column counts the engine file for a framework. For the **plain** engine it also counts what we wrote by hand in Modules 02 to 04: the agent loop, the tool descriptions, the agents and the router. The framework supplies those. (Nobody counts `tools.py`, the guards or the prompts: they are shared.)
5. **...and they charge for it in a different way.** The `Observe` column is code that exists *only* to report the framework's model calls into our traces. The plain engine needs none, because our wrapped client sees every call. Each framework needs its own hook.

So a framework trades **a lot less code** for **some extra waiting** and **a hook to write**. It does not change safety, and barely changes cost. That is a useful, honest summary, and you can only say it because you measured.

## Which would you pick?
We need a rule. Our first rule was written **before** we looked at the numbers: highest **safety**, then highest **correctness**, then lowest **cost**. (The plain engine is the baseline, so it is not a candidate.)

In [3]:
strict = scoreboard.pick_winner_strict(summaries)
print("The first rule picks:", strict["engine"])
print()
print("Engine".ljust(11) + "Cost per email".ljust(18) + "Average seconds")
costs = []
for item in summaries:
    if item["engine"] != "plain":
        print(item["engine"].ljust(11) + ("$" + format(item["avg_cost"], ".5f")).ljust(18) + format(item["avg_seconds"], ".1f"))
        costs.append(item["avg_cost"])

# How far apart are the two cheapest frameworks?
costs.sort()
gap = (costs[1] - costs[0]) / costs[0] * 100
print()
print("The two cheapest frameworks differ by only " + format(gap, ".1f") + "% in cost per email.")

The first rule picks: langgraph

Engine     Cost per email    Average seconds
sdk        $0.00121          4.9
langgraph  $0.00119          4.0
crewai     $0.00124          6.6

The two cheapest frameworks differ by only 1.4% in cost per email.


**Look at those costs.** Every framework is tied on safety and correctness, so the first rule falls through to the **cost**, and the cheapest wins. But the two cheapest differ by only a **few percent**, a hundredth of a cent per email. That is inside the run-to-run noise: run the scoreboard again and the order can flip. When we first ran it while writing this module, the first rule picked **CrewAI**, which was also the *slowest* engine, on exactly this kind of margin.

The flaw is in the **rule**, not in the numbers: *it never said what counts as a tie.* This is a common mistake. A rule that ranks by a number needs a **margin**, below which two numbers are treated as equal, so that a tiny difference cannot overrule a big one.

Our second rule says it explicitly. Cost and speed only count when two engines differ by **more than 10%**. If everything is still tied, the engine with **less code to maintain** wins.

*From `scoreboard.py`:*

```python
TIE = 0.10  # two numbers that differ by less than 10% count as a TIE


def pick_winner_strict(summaries):
    """Our FIRST rule: the highest safety score, then the highest pass rate, then the lowest cost.
    It never says what a tie is, so a difference of a hundredth of a cent decides. See notebook 5."""
    best = None
    best_key = None

    for item in summaries:
        if item["engine"] == "plain":
            continue
        key = (item["safety_passed"] / item["safety_total"], item["passed"] / item["total"], -item["avg_cost"])
        if best is None or key > best_key:
            best = item
            best_key = key
    return best


def clearly_different(first, second):
    """True when two numbers differ by more than the tie margin."""
    smaller = min(first, second)
    if smaller == 0:
        return first != second
    return abs(first - second) / smaller > TIE


def better_of(first, second):
    """Which of two engines is better? Safety first, then the pass rate. Then cost and speed, but ONLY when
    they are clearly different. If everything is still a tie, the one with less code to maintain wins."""
    first_safety = first["safety_passed"] / first["safety_total"]
    second_safety = second["safety_passed"] / second["safety_total"]
    if first_safety != second_safety:
        if first_safety > second_safety:
            return first
        return second

    first_rate = first["passed"] / first["total"]
    second_rate = second["passed"] / second["total"]
    if first_rate != second_rate:
        if first_rate > second_rate:
            return first
        return second

    if clearly_different(first["avg_cost"], second["avg_cost"]):
        if first["avg_cost"] < second["avg_cost"]:
            return first
        return second

    if clearly_different(first["avg_seconds"], second["avg_seconds"]):
        if first["avg_seconds"] < second["avg_seconds"]:
            return first
        return second

    if first["code_lines"] + first["observe_lines"] <= second["code_lines"] + second["observe_lines"]:
        return first
    return second


def pick_winner(summaries):
    """Choose a framework with the rule above. The plain engine is the baseline, so it is not a candidate."""
    best = None
    for item in summaries:
        if item["engine"] == "plain":
            continue
        if best is None:
            best = item
        else:
            best = better_of(best, item)
    return best
```

In [4]:
winner = scoreboard.pick_winner(summaries)
print("By the tie-aware rule, the framework to pick is:", winner["engine"])

By the tie-aware rule, the framework to pick is: langgraph


We **changed the rule after seeing the data**, and we said so. That is fine when you are honest about it and the new rule is reasonable on its own. It would *not* be fine to keep adjusting a rule until it picks your favourite. A good habit is to decide how you will judge **before** you measure, and to record any change, with the reason.

**A rule is not a verdict.** If two frameworks are close, the numbers cannot decide, and **your problem** should:

| If you need... | Lean towards | Because |
|---|---|---|
| The quickest start, with the fewest ideas | **OpenAI Agents SDK** | Handoffs, hooks and tracing out of the box, and the least code of the three |
| Control over the flow, branches, pausing for a person, saved state | **LangGraph** | You draw the flow, and the human exit is just a node |
| Work that reads like a team handing over tasks | **CrewAI** | Roles and tasks match how people describe a team |
| One agent, full visibility, the fastest answers | **Plain Python** | Nothing hidden and no hook to write, at the cost of writing the loop yourself |

And whatever you choose, **the next time you upgrade a framework, run this scoreboard again.** That is what the evals are for.

Next: **decide, ship, and look back.**